# Treino do modelo do sistema assistivo — ambientes internos (Kaggle)

Ajuste fino do **YOLO26 a partir dos pesos do COCO**, num dataset juntado de **38 classes de
ambiente interno**:

| Fonte | Classes |
|---|---|
| COCO (8.000 imagens) | 29 classes: pessoa, cadeira, sofá, cama, mesa, banco, vaso sanitário, pia, geladeira, fogão, vaso de planta, TV, gato, cachorro, mochila, bolsa, mala, celular, garrafa, copo, notebook, guarda-chuva, micro-ondas, controle remoto, livro, relógio, teclado, mouse, vaso |
| Accessibility Barriers | **escada, degrau, corrimão, rampa** |
| HomeObjects-3K + DoorDetect | **porta, guarda-roupa, luminária, janela** (+ cama, TV, notebook...) |
| ROD | **lixeira**, escada |
| Imagens da ESP32-CAM (opcional) | a câmera de verdade |

## Como rodar (uma vez)

1. **Antes de tudo, envie o código ao GitHub** (`git push`): este notebook clona o repositório.
2. kaggle.com → **Create → New Notebook** → **File → Import Notebook** → este arquivo.
3. **Verifique o telefone** em kaggle.com → avatar → *Settings* → *Phone Verification*: sem isso o
   Kaggle não libera internet nem GPU ("Permission 'kernelSessions.enableInternet' was denied").
   Depois, no painel à direita (**Session options**):
   - **Accelerator: GPU T4 x2**;
   - **Internet: On**.
4. *(Opcional)* imagens da ESP: **Add Input → Upload**, envie o `esp_revisado.zip` exportado do CVAT
   (*YOLO 1.1*, **com as imagens**) como dataset **privado**. O notebook encontra sozinho.
5. **Save Version → Save & Run All (Commit)**. Pode fechar o navegador: roda no servidor do Kaggle.
   Leva **~5 a 6 h** (limite de 12 h por execução; a cota é de ~30 h de GPU por semana).
6. Quando terminar, abra a versão salva → aba **Output** → baixe `assistivo-interno.pt` e os
   `resultado_*.json`.

Rostos nas imagens da ESP: mantenha o dataset do Kaggle **privado** (LGPD).

In [ ]:
# 1. GPU e disco
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | cat

In [ ]:
# 2. Configuração
import glob
import os

import torch

SAIDA = '/kaggle/working'      # o que fica salvo na versão (limite de 20 GB)
TMP = '/tmp'                   # downloads e datasets montados (não vão para a saída)
CACHE = f'{TMP}/cache'

# Tamanho do modelo: 's' (padrão), 'm' ou 'l'. Maior acerta mais (COCO: s 49,0 / m 53,2 / l 56,7
# de mAP50-95) e treina mais devagar (m ~2x, l ~3x o s). Na GTX 1650: s ~17 ms, l ~41 ms por quadro.
TAMANHO = 's'
GPUS = torch.cuda.device_count()
DISPOSITIVOS = list(range(GPUS)) if GPUS > 1 else 0
LOTE = (8 if TAMANHO == 'l' else 16) * max(1, GPUS)   # lote por GPU x número de GPUs
EPOCAS, EPOCAS_PROFESSOR = 40, 20
IMAGENS_COCO, IMAGENS_ROD = 8000, 2000
NOME = 'assistivo-interno'

# Imagens da ESP (opcional): uma pasta em /kaggle/input com a exportação do CVAT.
marcas = glob.glob('/kaggle/input/**/obj.names', recursive=True) + \
         glob.glob('/kaggle/input/**/data.yaml', recursive=True)
ESP = os.path.dirname(marcas[0]) if marcas else None
SPLIT = 'test' if ESP else 'val'   # com a ESP, a avaliação é no teste da ESP (sessões inéditas)

print(f'GPUs: {GPUS} | modelo: yolo26{TAMANHO} | lote: {LOTE} | imagens da ESP: {ESP or "não"}')

In [ ]:
# 3. Código do projeto e dependências (a mesma versão do ultralytics das medições)
!rm -rf /tmp/deep-vision && git clone -q https://github.com/JeanLimaa/deep-vision.git /tmp/deep-vision
!pip install -q "ultralytics==8.4.160" "pydantic-settings>=2.6"
%cd /tmp/deep-vision/server
!python -c "import sys; sys.path.insert(0, 'training'); import build_dataset as b; print(len(b.VOCAB), 'classes:', b.VOCAB)"

## 4. Professor das classes novas (~40 min)

Cada dataset rotulou só as próprias classes: a porta que aparece numa foto do dataset de escadas
está **sem rótulo**, e isso ensinaria ao modelo que porta é fundo. Este "professor" aprende só as
9 classes novas (escada, degrau, porta, lixeira, corrimão, rampa, guarda-roupa, luminária, janela);
na célula 5 ele preenche essas classes nas imagens das outras fontes.

In [ ]:
from ultralytics import YOLO

!python training/build_dataset.py build --stage professor --out {TMP}/professor --cache {CACHE} --rod {IMAGENS_ROD} --overwrite
YOLO('yolo26s.pt').train(data=f'{TMP}/professor/data.yaml', epochs=EPOCAS_PROFESSOR, imgsz=640,
                         batch=16 * max(1, GPUS), device=DISPOSITIVOS, workers=4, patience=8,
                         project=f'{SAIDA}/runs', name='professor-interno', exist_ok=True)
PROFESSOR = f'{SAIDA}/runs/professor-interno/weights/best.pt'

## 5. Dataset (~1 h)

Junta tudo. Rótulos automáticos só **acrescentam** objetos das classes que cada fonte não rotulou;
rótulo humano nunca é substituído. As **8.000 imagens do COCO** são a "revisão" das 29 classes do
COCO: sem elas, o ajuste fino as esqueceria. 30% das imagens públicas de treino são degradadas como
a OV2640 (escuro, borrão, JPEG forte, baixa resolução, cor). Com as imagens da ESP, o **teste** é só
de sessões que não entram no treino. Detalhes em `manifest.csv` e `SOURCES.md` (licenças).

In [ ]:
esp = f'--esp "{ESP}"' if ESP else ''
!python training/build_dataset.py build {esp} --out {TMP}/assistivo --cache {CACHE} --coco {IMAGENS_COCO} --rod {IMAGENS_ROD} --teachers yolo26l.pt yolo26l-objv1-150.pt "{PROFESSOR}" --overwrite
!cp {TMP}/assistivo/manifest.csv {TMP}/assistivo/SOURCES.md {TMP}/assistivo/data.yaml {SAIDA}/

## 6. Treino (~3 a 4 h com 2 GPUs)

Parte dos pesos do COCO: o ultralytics casa as 29 classes do COCO pelo nome e reaproveita a camada de
classificação delas; só as 9 classes novas começam do zero. `save_period=5` guarda um ponto de
retomada a cada 5 épocas na saída.

In [ ]:
YOLO(f'yolo26{TAMANHO}.pt').train(
    data=f'{TMP}/assistivo/data.yaml', epochs=EPOCAS, imgsz=640, batch=LOTE, device=DISPOSITIVOS,
    workers=4, patience=15, cos_lr=True, close_mosaic=10, hsv_v=0.5, degrees=5,
    project=f'{SAIDA}/runs', name=NOME, exist_ok=True, save_period=5)

## 7. Avaliação

Modelo pré-treinado (`yolo26l`, COCO) contra o modelo treinado, **nas mesmas classes**
(`--same-classes`): sem isso, o modelo COCO é punido por achar objetos reais que o conjunto não
rotula. Medido como o servidor mede (limiar 0,35), na imagem original, com pouca luz e com borrão de
movimento. O `val` do ultralytics dá o mAP oficial por classe.

In [ ]:
import json
import subprocess

TREINADO = f'{SAIDA}/runs/{NOME}/weights/best.pt'
QUADRO_DA_PLACA = ['--as-is'] if ESP else []  # imagens da ESP já são o JPEG da placa


def avaliar(nome, pesos):
    saida = f'{SAIDA}/resultado_{nome}.json'
    cmd = ['python', 'training/pipeline_ablation.py', '--data', f'{TMP}/assistivo',
           '--split', SPLIT, *QUADRO_DA_PLACA, '--only', 'original', 'escuro', 'borrao',
           '--names', f'{TMP}/assistivo/data.yaml', '--same-classes', '--weights', pesos,
           '--device', 'cuda', '--json', saida]
    print(f'== {nome}')
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'ERRO: ' + r.stderr[-3000:])


avaliar('pre_treinado', 'yolo26l.pt')
avaliar('treinado', TREINADO)

metricas = YOLO(TREINADO).val(data=f'{TMP}/assistivo/data.yaml', split=SPLIT, device=0,
                              project=f'{SAIDA}/runs', name=f'{NOME}_{SPLIT}', exist_ok=True)
por_classe = {metricas.names[int(c)]: {'AP50': round(float(metricas.box.ap50[i]), 3),
                                       'AP50-95': round(float(metricas.box.ap[i]), 3)}
              for i, c in enumerate(metricas.box.ap_class_index)}
resumo = {'mAP50': round(float(metricas.box.map50), 3),
          'mAP50-95': round(float(metricas.box.map), 3), 'por_classe': por_classe}
with open(f'{SAIDA}/resultado_val_por_classe.json', 'w') as f:
    json.dump(resumo, f, indent=1, ensure_ascii=False)
print(f"mAP50 = {resumo['mAP50']}   mAP50-95 = {resumo['mAP50-95']}")
for nome, v in sorted(por_classe.items(), key=lambda kv: -kv[1]['AP50']):
    print(f"  {nome:15s} AP50 {v['AP50']:.3f}   AP50-95 {v['AP50-95']:.3f}")

## 8. Exportar para o servidor

Baixe `assistivo-interno.pt` da aba **Output** para a pasta `models/` do projeto e, no `server/.env`:

```
AVS_VISION__MODEL_PATH=assistivo-interno.pt
```

**Sem** `AVS_VISION__EXTRA_MODEL_PATHS`: o modelo treinado já tem todas as classes.

In [ ]:
import shutil

shutil.copy(TREINADO, f'{SAIDA}/{NOME}.pt')
print('Pronto:', f'{SAIDA}/{NOME}.pt')